# EfficientNetB0 only — push test accuracy to ≥ 80%

**One model. One T4 runtime. ~30–45 minutes.** Do not rerun VGG / ResNet / MobileNet / YOLO.

Current Drive EfficientNetB0 is **76.5%** test. Weak classes: truck, cup, chair, car.

This notebook:
1. Reloads `SmartVision_artifacts/models/efficientnetb0.keras`
2. Unfreezes most of the backbone (BatchNorm stays frozen)
3. Trains with **class weights** (extra weight on truck / cup / chair / car) at a low LR
4. Reports plain test accuracy **and** horizontal-flip TTA

Copies `efficientnetb0_run_pre80.keras` before overwriting so you can roll back.

If Colab died: reconnect, Run all — it resumes from the latest `efficientnetb0.keras` on Drive.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install tensorflow pandas scikit-learn matplotlib seaborn pillow


In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications, optimizers, callbacks

IN_COLAB = "google.colab" in sys.modules
print("TF", tf.__version__, "GPU", tf.config.list_physical_devices("GPU"))
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    if zip_path.exists() and not (data_dir / "classification" / "train").exists():
        import zipfile
        print("Unzipping dataset zip...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_dir if not any(data_dir.glob("*")) else PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    OUT_ROOT = PROJECT_ROOT

CLASS_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
    "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake", "chair",
    "couch", "potted plant", "bed",
]
NUM_CLASSES = 25
IMAGE_SIZE = 224
BATCH = 16

DATA = PROJECT_ROOT / "smartvision_dataset" / "classification"
if not (DATA / "train").exists():
    alt = Path("/content/drive/MyDrive/smartvision_dataset/classification")
    if (alt / "train").exists():
        DATA = alt
print("DATA", DATA, (DATA / "train").exists())

MODELS_DIR = OUT_ROOT / "models"
REPORTS = OUT_ROOT / "reports"
FIGURES = REPORTS / "figures"
for p in (MODELS_DIR, REPORTS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

src = MODELS_DIR / "efficientnetb0.keras"
assert src.exists(), src
bak = MODELS_DIR / "efficientnetb0_run_pre80.keras"
if not bak.exists():
    shutil.copy2(src, bak)
    print("Backed up current EfficientNet ->", bak)
print("Will fine-tune", src, "size MB", round(src.stat().st_size / 1e6, 1))


In [ ]:
def list_image_label(split):
    paths, labels = [], []
    for idx, name in enumerate(CLASS_NAMES):
        for f in sorted((DATA / split / name).glob("*.jpg")):
            paths.append(str(f)); labels.append(idx)
    return tf.constant(paths), tf.constant(labels, dtype=tf.int32)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    return tf.cast(img, tf.float32), label

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15.0 / 360.0, fill_mode="nearest"),
    layers.RandomBrightness(0.20),
    layers.RandomContrast(0.20),
    layers.RandomZoom(0.20),
    layers.RandomTranslation(0.08, 0.08),
], name="aug")

def color_jitter(img, label):
    img = tf.image.random_saturation(img / 255.0, 0.7, 1.3) * 255.0
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_ds(split, training=False):
    paths, labels = list_image_label(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(color_jitter, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE), np.array(labels)

train_ds, y_train = make_ds("train", training=True)
val_ds, y_val = make_ds("val", training=False)
test_ds, y_test = make_ds("test", training=False)
print("n", len(y_train), len(y_val), len(y_test))

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
# extra boost on the classes EfficientNet still confuses
boost = {"truck": 1.6, "cup": 1.5, "chair": 1.4, "car": 1.3}
for name, b in boost.items():
    cw[CLASS_NAMES.index(name)] *= b
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("highest class weights", sorted(zip(CLASS_NAMES, cw), key=lambda t: -t[1])[:6])

def mixup_ds(ds, alpha=0.2):
    def _mix(bx, by):
        g1 = tf.random.gamma([], alpha); g2 = tf.random.gamma([], alpha)
        lam = g1 / (g1 + g2)
        idx = tf.random.shuffle(tf.range(tf.shape(bx)[0]))
        mx = lam * bx + (1.0 - lam) * tf.gather(bx, idx)
        y1 = tf.one_hot(by, NUM_CLASSES)
        y2 = tf.one_hot(tf.gather(by, idx), NUM_CLASSES)
        return mx, lam * y1 + (1.0 - lam) * y2
    return ds.map(_mix, num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
keras.mixed_precision.set_global_policy("float32")
model = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")

backbone = None
for layer in model.layers:
    if "efficientnet" in layer.name.lower():
        backbone = layer
        break
assert backbone is not None, [l.name for l in model.layers]

backbone.trainable = True
n_unfreeze = 180
keep = max(0, len(backbone.layers) - n_unfreeze)
for i, layer in enumerate(backbone.layers):
    layer.trainable = i >= keep
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print("trainable", sum(l.trainable for l in backbone.layers), "/", len(backbone.layers))

try:
    loss = keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1)
except TypeError:
    loss = "sparse_categorical_crossentropy"

model.compile(
    optimizer=optimizers.Adam(2e-5),
    loss=loss,
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")],
)

ckpt = callbacks.ModelCheckpoint(
    str(MODELS_DIR / "efficientnetb0.keras"),
    monitor="val_accuracy", save_best_only=True, mode="max", verbose=1,
)
hist = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=CLASS_WEIGHT,
    callbacks=[
        ckpt,
        callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ],
    verbose=1,
)
model = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
print("stage A finished")


In [ ]:
def eval_acc(model, tta=False):
    y_true, y_pred = [], []
    for xb, yb in test_ds:
        p = model.predict(xb, verbose=0)
        if tta:
            p = 0.5 * (p + model.predict(tf.image.flip_left_right(xb), verbose=0))
        y_true.append(yb.numpy())
        y_pred.append(p.argmax(axis=1))
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = float(accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    print("accuracy", round(acc, 4), "TTA" if tta else "plain")
    return acc

acc = eval_acc(model, tta=False)
acc_tta = eval_acc(model, tta=True)
print("PLAIN", acc, "TTA(hflip)", acc_tta)
print("RUBRIC plain >= 0.80?", acc >= 0.80)


In [ ]:
# If still under 80%, one short extra pass: open the rest of the backbone, tiny LR
if acc < 0.80:
    print("Still under 80% — extra pass, unfreeze remaining conv layers, lr=8e-6, 12 epochs")
    model = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
    backbone = next(l for l in model.layers if "efficientnet" in l.name.lower())
    backbone.trainable = True
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(
        optimizer=optimizers.Adam(8e-6),
        loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")],
    )
    model.fit(
        train_ds, validation_data=val_ds, epochs=12, class_weight=CLASS_WEIGHT,
        callbacks=[
            callbacks.ModelCheckpoint(str(MODELS_DIR / "efficientnetb0.keras"),
                                      monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
            callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, mode="max"),
        ],
        verbose=1,
    )
    model = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
    acc = eval_acc(model, tta=False)
    acc_tta = eval_acc(model, tta=True)
    print("AFTER EXTRA  PLAIN", acc, "TTA", acc_tta, "rubric", acc >= 0.80)
else:
    print("Already >= 80% plain test. Stop.")

print("Copy Drive efficientnetb0.keras into the GitHub models/ folder if plain acc improved.")
print("Roll back with efficientnetb0_run_pre80.keras if this run is worse.")
